In [1]:
import polars as pl
import duckdb
from pathlib import Path

pl.Config.set_fmt_str_lengths(100)
data_path = Path("../data")

In [2]:
# Read available files
datasets = pl.read_csv(data_path / "Datasets 9424.csv")

# Read kg files
conn = duckdb.connect(data_path / "event_datasets.duckdb")
datasets_id = conn.query("FROM kg_datasets SELECT *").pl()
datasets_vars = conn.query("FROM kg_variables SELECT *").pl()

In [3]:
# Merge
df = (
    datasets
    .join(datasets_id, left_on="Bestandsnaam", right_on="alt_title", how="left")
)

In [4]:
# No matching to KG (11 real datasets that are not matched)
(
    df
    .filter(pl.col("is_frequency_dataset").is_null())
    ["Bestandsnaam"]
    .to_list()
)

['9424_LISS_background_survey_merged_for_importCBKV1',
 '9424_LISS_core_studies_long_updCBKV1.csv',
 '9424_LISS_core_studies_longCBKV1',
 '9424_ZYGV2_942420250724CBKV1',
 'EIGENDOMTAB',
 'EIGENDOMWOZBAGTAB',
 'EIGENDOMWOZTAB',
 'GBAPERSOONKTAB',
 'GBASCHEIDINGENMASSATAB',
 'KINDEROPVANGLOCATIES',
 'LISS',
 'NCOTOOL',
 'NIETVSLGTAB',
 'VEHTAB',
 'VRLGBAPERSOONKTAB',
 'VSLGTAB',
 'WOZ']

In [5]:
#Several IDs per dataset (19 datasets, including SPOLIS (next cell))
(
    df
    .filter(pl.col("Bestandsnaam").is_duplicated())
    .select(["Bestandsnaam"])
    .unique()
    .shape
)

(19, 1)

In [6]:
# 11 different IDs for SPOLIS
(
    df
    .filter(pl.col("Bestandsnaam") == "SPOLISBUS")
    .select("Bestandsnaam", "dataset_id", "valid_from", "valid_until")
    .sort(by="valid_from")
)

Bestandsnaam,dataset_id,valid_from,valid_until
str,str,date,date
"""SPOLISBUS""","""https://doi.org/10.57934/0B01E4108024847A""",2010-01-01,2015-01-01
"""SPOLISBUS""","""https://doi.org/10.57934/0B01E410802AE699""",2015-01-01,2016-01-01
"""SPOLISBUS""","""https://doi.org/10.57934/0B01E410802EE816""",2016-01-01,2016-11-22
"""SPOLISBUS""","""https://doi.org/10.57934/0B01E41080337ACE""",2016-11-22,2018-03-22
"""SPOLISBUS""","""https://doi.org/10.57934/0B01E410803922C0""",2018-03-22,2020-02-12
…,…,…,…
"""SPOLISBUS""","""https://doi.org/10.57934/0B01E410805FD44A""",2021-03-16,2022-03-15
"""SPOLISBUS""","""https://doi.org/10.57934/0B01E4108071EA4C""",2022-03-15,2023-02-17
"""SPOLISBUS""","""https://doi.org/10.57934/0B01E41080767395""",2023-02-17,2024-02-17


In [7]:
# Variables of SPOLIS per edition
df_spolis = (
    df
    .filter(pl.col("Bestandsnaam") == "SPOLISBUS")
    .sort(by="valid_from")
    .join(datasets_vars, on="dataset_id", how="left")
)

In [8]:
# Different number of variables
(
    df_spolis
    .group_by(["dataset_id","valid_from","valid_until"])
    .len("variable_name")
    .sort(by="valid_from")
)

dataset_id,valid_from,valid_until,variable_name
str,date,date,u32
"""https://doi.org/10.57934/0B01E4108024847A""",2010-01-01,2015-01-01,184
"""https://doi.org/10.57934/0B01E410802AE699""",2015-01-01,2016-01-01,186
"""https://doi.org/10.57934/0B01E410802EE816""",2016-01-01,2016-11-22,200
"""https://doi.org/10.57934/0B01E41080337ACE""",2016-11-22,2018-03-22,212
"""https://doi.org/10.57934/0B01E410803922C0""",2018-03-22,2020-02-12,220
…,…,…,…
"""https://doi.org/10.57934/0B01E410805FD44A""",2021-03-16,2022-03-15,248
"""https://doi.org/10.57934/0B01E4108071EA4C""",2022-03-15,2023-02-17,266
"""https://doi.org/10.57934/0B01E41080767395""",2023-02-17,2024-02-17,272


In [9]:
# Comparing the last two (same number of variables)
df1 = df_spolis.filter(pl.col("dataset_id") == "https://doi.org/10.57934/0B01E410807C72A7").unique()
df2 = df_spolis.filter(pl.col("dataset_id") == "https://doi.org/10.57934/0B01E4108081B831").unique()

print("Present only in 1", set(df1["variable_name"]) - set(df2["variable_name"]))
print("Present only in 2", set(df2["variable_name"]) - set(df1["variable_name"]))

Present only in 1 set()
Present only in 2 set()


In [10]:
# Some variables have several IDs even within the same dataset
df1.filter(pl.col("variable_name")=="SIndAvrLkvAgWn")

Bestandsnaam,Bestandsonderwerp,Type bestand,Status,dataset_id,title,publication_date,valid_from,valid_until,frequency,keywords,sampling_procedure,is_event_dataset,is_frequency_dataset,variable_id,variable_name,label_,vocab_label,description,data_type,definition,num_predicates,tijdstip_predicates,valid_from_right,is_tijdstip
str,str,str,str,str,str,date,date,date,str,list[str],str,bool,bool,str,str,str,str,str,str,str,str,list[str],date,bool
"""SPOLISBUS""","""SPOLIS""","""Catalogusbestand""","""Available""","""https://doi.org/10.57934/0B01E410807C72A7""","""Banen en lonen op basis van de Polisadministratie""",null,2024-02-17,2025-01-01,"""Niet eenduidig""","[""Arbeid en sociale zekerheid""]","""In de registratie van de Polisadministratie vallen alle inkomstenverhoudingen in Nederland die onder…",true,true,"""https://kg.odissei.nl/.well-known/genid/74d1d1ad3de476dbd31861cadc247c7b""","""SIndAvrLkvAgWn""","""Indicatie aanvraag loonkostenvoordeel arbeidsgehandicapte werknemer""","""Indicatie aanvraag loonkostenvoordeel arbeidsgehandicapte werknemer""","""De inhoudingsplichtige/werkgever kan onder de volgende voorwaarden in aanmerking komen voor het loon…","""String""","""Indicatie of de administratieve eenheid heeft verzocht in aanmerking te komen voor een loonkostenvoo…","""14""",[],2018-02-09,false
"""SPOLISBUS""","""SPOLIS""","""Catalogusbestand""","""Available""","""https://doi.org/10.57934/0B01E410807C72A7""","""Banen en lonen op basis van de Polisadministratie""",null,2024-02-17,2025-01-01,"""Niet eenduidig""","[""Arbeid en sociale zekerheid""]","""In de registratie van de Polisadministratie vallen alle inkomstenverhoudingen in Nederland die onder…",true,true,"""https://kg.odissei.nl/.well-known/genid/5f12747729c5792bd4c5b4a5a6904133""","""SIndAvrLkvAgWn""","""Indicatie aanvraag loonkostenvoordeel arbeidsgehandicapte werknemer""","""Indicatie aanvraag loonkostenvoordeel arbeidsgehandicapte werknemer""","""De inhoudingsplichtige/werkgever kan onder de volgende voorwaarden in aanmerking komen voor het loon…","""String""","""Indicatie of de administratieve eenheid heeft verzocht in aanmerking te komen voor een loonkostenvoo…","""14""",[],2018-02-09,false


In [11]:
# See what's changed
df_combined = df1.join(df2, on="variable_name", suffix="_new")
vars_updated = []
for var in df1.columns:
    if var != "variable_name":
        df_f = df_combined.filter(pl.col(var) != pl.col(f"{var}_new"))
        if df_f.shape[0] > 0:
            print(var, df_f.shape[0])
            vars_updated.append(var)

dataset_id 548
valid_from 548
variable_id 548
description 52


In [12]:
# Basically no differences (some commas, quotes, etc)
var = "description"
df_combined.filter(pl.col(var) != pl.col(f"{var}_new")).select(["variable_name","description","description_new"])

variable_name,description,description_new
str,str,str
"""CdCaoInl""","""Deze code mag alleen ingevuld worden bij payrollers. Geef bij samenloop van meerdere inleners in ee…","""Deze code mag alleen ingevuld worden bij payrollers. Geef bij samenloop van meerdere inleners in ee…"
"""CdCaoInl""","""Deze code mag alleen ingevuld worden bij payrollers. Geef bij samenloop van meerdere inleners in ee…","""Deze code mag alleen ingevuld worden bij payrollers. Geef bij samenloop van meerdere inleners in ee…"
"""SPRWAOAOF""","""Deze premie is onderdeel van de werknemersverzekeringen. Het loontijdvak is de periode tussen SDATUM…","""Deze premie is onderdeel van de werknemersverzekeringen. Het loontijdvak is de periode tussen SDATUM…"
"""SPRWAOAOF""","""Deze premie is onderdeel van de werknemersverzekeringen. Het loontijdvak is de periode tussen SDATUM…","""Deze premie is onderdeel van de werknemersverzekeringen. Het loontijdvak is de periode tussen SDATUM…"
"""SOPGRCHTVAKBSL""","""Het loontijdvak is de periode tussen SDATUMAANVANGIKO en SDATUMEINDEIKO. Het opgebouwde recht op vak…","""Het loontijdvak is de periode tussen SDATUMAANVANGIKO en SDATUMEINDEIKO. Het opgebouwde recht op vak…"
…,…,…
"""SCDAARD""","""Als de werknemer niet is verzekerd voor de werknemersverzekeringen (zoals meewerkende kinderen) geld…","""Als de werknemer niet is verzekerd voor de werknemersverzekeringen (zoals meewerkende kinderen) geld…"
"""SCAO""","""Dit is de code die het Ministerie van Sociale Zaken en Werkgelegenheid of het CBS heeft vastgesteld.…","""Dit is de code die het Ministerie van Sociale Zaken en Werkgelegenheid of het CBS heeft vastgesteld.…"
"""SCAO""","""Dit is de code die het Ministerie van Sociale Zaken en Werkgelegenheid of het CBS heeft vastgesteld.…","""Dit is de code die het Ministerie van Sociale Zaken en Werkgelegenheid of het CBS heeft vastgesteld.…"
